<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/iLogos/logo_novafct.png" style="width: 30%;">

# Departamento de Engenharia Mecânica e Industrial
## Mecânica dos Sólidos II

## Estabilidade de colunas

### Problema 3

A figura representa uma treliça de 7 barras apoiada nos pontos A e C e que suporta a carga concentrada $P$ = 200 kN no ponto B. Sabendo que é construída com perfis tubulares de aço ($E =$ 210 GPa) de espessura $t =$ 6 mm, determine o diâmetro mínimo de cada barra, considerando que a tensão admissível do material das barras é $\sigma_\textrm{adm}=$ 355 MPa e que a carga crítica nas barras à compressão não pode ser ultrapassada.

<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/Notebooks/Au12/P3/MSII_Au12_P3.png"
style="width: 30%;"/>

In [1]:
import numpy as np
import sympy as sy
from sympy.solvers import solve
from typing import NamedTuple

# Dados
# unidades SI: Pa, N, m 
class dados(NamedTuple):
    P : float 
    L : float
    ang: float
    t : float
    E : float
    sadm : float
    
var = dados(200.e3, 5., 60., 6.e-3, 210.e9, 355e6)
print(f'{var = }')

ADy = np.tan(np.deg2rad(var.ang))*(var.L/2)
print(f'{ADy = :.2f} m')
AD = (var.L/2)/np.cos(np.deg2rad(var.ang))
print(f'{AD = :.2f} m')

c = np.cos(np.deg2rad(var.ang))
s = np.sin(np.deg2rad(var.ang))

var = dados(P=200000.0, L=5.0, ang=60.0, t=0.006, E=210000000000.0, sadm=355000000.0)
ADy = 4.33 m
AD = 5.00 m


### Resolução

#### Análise do equilíbrio estático

Pela análise do equilíbrio global da estrutura, determinam-se as reações ilustradas no diagrama de corpo livre.

In [2]:
ay, cy = sy.symbols('ay cy')

# Equilíbrio
sumFy = ay + cy - var.P
# Momento em A
sumMA = cy*2*var.L - var.P*var.L

# Solução
sol = solve([sumFy, sumMA], (ay, cy))
Ay = sol[ay]
Cy = sol[cy]
print(f'Ay = {Ay*1e-3:.1f} kN')
print(f'Cy = {Cy*1e-3:.1f} kN')

Ay = 100.0 kN
Cy = 100.0 kN


#### Método dos nós

Considerando a simetria da estrutura e definindo uma estratégia para a aplicação do método dos nós, começando pelo nó A seguido do nó D:

##### Nó A

In [3]:
fad, fab = sy.symbols('fad fab')

# Equilíbrio
sumFx = fad*c + fab
sumFy = Ay + fad*s

# Solução
sol = solve([sumFx,sumFy], (fad, fab))
FAD = sol[fad]
FAB = sol[fab]
print(f'FAD = {FAD*1e-3:.1f} kN')
print(f'FAB = {FAB*1e-3:.1f} kN')

FAD = -115.5 kN
FAB = 57.7 kN


##### Nó D

In [4]:
fde, fdb = sy.symbols('fde fdb')

# Equilíbrio
sumFx = -FAD*c + fde + fdb*c
sumFy = -FAD*s - fdb*s

# Solução
sol = solve([sumFx,sumFy], (fde, fdb))
FDE = sol[fde]
FDB = sol[fdb]
print(f'FDE = {FDE*1e-3:.1f} kN')
print(f'FDB = {FDB*1e-3:.1f} kN')

FDE = -115.5 kN
FDB = 115.5 kN


Pela análise dos esforços axiais nas barras, conclui-se que as barras mais solicitadas são:

- AD, DE, EC (Compressão)
- AB, BC, DB, BE (Tração)
-
<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/Notebooks/Au12/P3/MSII_Au12_P3_Ftool.png"
style="width: 30%;"/>

## Procedendo ao dimensionamento das barras à tração e compressão:

### **Barras à tração**

As barras á tração poderão ceder por plastificação. A inequação de dimensioamento pode ser escrita por.

\begin{equation*}
\sigma_\textrm{adm} \ge \frac{N}{A}
\end{equation*}

com,

\begin{equation*}
A = \pi (r_e² - r_i²)
\quad\textrm{e}\quad
t = r_e - r_i
\end{equation*}

É possível resolver um sistema de equações para determinar $r_e$ e  $r_i$ que satisfazem o dimensionamento.


In [5]:
a = sy.symbols('a')

N = FDB 
eqT = var.sadm - N/a
sol = solve(eqT, a)
At = sol[0]
print(f'{At = :.3e} m²')

At = 3.253e-4 m²


In [6]:
re, ri = sy.symbols('re ri')

eq1 = At - np.pi*(re**2 - ri**2) 
eq2 = var.t - re + ri 

sol = solve((eq1,eq2), (re,ri)) # Solve the system of equations
RE = sol[0][0]
RI = sol[0][1]
print(f're = {RE:.4f} m ({RE*1e3:.1f} mm) | ri = {RI:.4f} m ({RI*1e3:.1f} mm)')
print(f'de = {2*RE:.4f} m ({2*RE*1e3:.1f} mm) | di = {2*RI:.4f} m ({2*RI*1e3:.1f} mm)')

re = 0.0116 m (11.6 mm) | ri = 0.0056 m (5.6 mm)
de = 0.0233 m (23.3 mm) | di = 0.0113 m (11.3 mm)


\begin{equation*}
A = \pi [(ri +t))² - r_i²]
\quad\therefore\quad
r_i = (A/\pi - t^2)/2t
\end{equation*}

In [7]:
ri_  = (At /np.pi - var.t**2)/(2*var.t)
print(f'(equação) ri = {ri_*1e3 :.2f} mm')

(equação) ri = 5.63 mm


### **Barras à compressão**

As barras á compressão poderão ceder por instabilidade elástica ou plastificação. 

Considerando as condiçẽos de fronteira,

\begin{equation*}
L_e = L
\end{equation*}

a condição de dimensionamento para evitar a instabilidade elástica é que a tensão normal aplicada não exceda a tensão crítica de encurvadura, ou seja:


\begin{equation*}
\sigma_{\textrm{crit}} \ge \frac{N}{A}
\quad\wedge\quad
\sigma_{\textrm{crit}} = \frac{P_\textrm{crit}}{A}
\quad\wedge\quad
\frac{\pi^2 EI}{AL^2} \ge \frac{N}{A}
\end{equation*}

simplificando,

\begin{equation*}
\frac{\pi^2 EI}{L^2} \ge N
\qquad\therefore\qquad I = f(A,N,L,E)
\end{equation*}

In [8]:
i = sy.symbols('i')

eq = np.pi**2*var.E*i / var.L**2 - N
sol = solve(eq,i)
Isec2 = sol[0]
print(f'Isec2 = {Isec2:.3e} m⁴')

Isec2 = 1.393e-6 m⁴


É possível agora resolver o sistema de equações para determinar $r_e$ e  $r_i$ que satisfazem o dimensionamento para a compressão:

\begin{equation*}
I = \frac{\pi}{4}(r_e^4 - r_i^4)
\quad\wedge\quad
t = r_e - r_i
\end{equation*}

In [9]:
eq3 = Isec2 - (np.pi/4)*(re**4 - ri**4) 
eq4 = var.t - re + ri 

sol = solve((eq3,eq4), (re,ri)) # Solve the system of equations
RE_2 = sol[0][0]
RI_2 = sol[0][1]
print(f're (2) = {RE_2:.4f} m ({RE_2*1e3:.1f} mm) | ri (2) = {RI_2:.4f} m ({RI_2*1e3:.1f} mm)')
print(f'de (2) = {2*RE_2:.4f} m ({2*RE_2*1e3:.1f} mm) | di (2) = {2*RI_2:.4f} m ({2*RI_2*1e3:.1f} mm)')

re (2) = 0.0449 m (44.9 mm) | ri (2) = 0.0389 m (38.9 mm)
de (2) = 0.0898 m (89.8 mm) | di (2) = 0.0778 m (77.8 mm)


In [10]:
re_dimensionado = max(RE, RE_2)
print(f're_dimensionado = max({RE*1e3:.1f}, {RE_2*1e3:.1f}) :: {re_dimensionado*1e3:.1f} mm')
print(f'dimensionado :: diâmetro \u2265 {2*re_dimensionado*1e3:.1f} mm')
print(f'\n Para as barras em tração o diâmetro de {2*RE*1e3:.1f} mm seria suficiente, mas para as barras sm compressão seria necessário um diâmetro maior de {2*re_dimensionado*1e3:.1f} mm.')

re_dimensionado = max(11.6, 44.9) :: 44.9 mm
dimensionado :: diâmetro ≥ 89.8 mm

 Para as barras em tração o diâmetro de 23.3 mm seria suficiente, mas para as barras sm compressão seria necessário um diâmetro maior de 89.8 mm.


---

Copyright (c) DEMI - FCT NOVA

Interactive computing by <a href="https://jupyter.org/" target="_blank"> <span
style="color:#333399"> Jupyter Notebook </span> </a> &nbsp;|&nbsp;Coded by <a href = "mailto: jmc.xavier@fct.unl.pt">José Xavier</a>

Licensed under  <a href="http://creativecommons.org/licenses/by-sa/4.0/"
target="_blank"> <span style="color:#333399;font-size: 20px"> CC BY-SA 4.0  </span></a>